# Assisgnment is: 
1. take a multiple pdf with text,image,table, fetch the data from pdf at least there should be 200 pages
2. use the sementic chunking technique
    required do chunking and then embedding
4. store it inside the vector database(use any of them 1. mongodb 2. astradb 3. opensearch 4.milvus) ## i have not discuss then you n eed to explore
5. create a index with all three index machnism(Flat, HNSW, IVF)## i have not discuss then you need to explore
6. create a retriever pipeline
7. check the retriever time(which one is fastet)
8. print the accuray score of every similarity search
9. perform the reranking either using BM25 or MMR ## i have not discuss then you need to explore
10. then write a prompt template
11. generte a oputput through llm
12. render that output over the DOCx ## i have not discuss then you need to explore


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()  #load all the environment variables

#https://python.langchain.com/docs/integrations/text_embedding/

True

In [2]:
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")
os.environ['GEMINI_API_KEY']=os.getenv("GEMINI_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

## load pdf data

In [3]:
!wget --user-agent "Mozilla" "https://arxiv.org/pdf/2307.09288.pdf" -O "/Users/jatinder.singh/drive/git/Agentic_AI/agentic_ai/Multimodel_RAG_report_analysis/llama2.pdf"

--2025-07-04 17:19:22--  https://arxiv.org/pdf/2307.09288.pdf
Resolving arxiv.org (arxiv.org)... 151.101.67.42, 151.101.195.42, 151.101.131.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.67.42|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: http://arxiv.org/pdf/2307.09288 [following]
--2025-07-04 17:19:22--  http://arxiv.org/pdf/2307.09288
Connecting to arxiv.org (arxiv.org)|151.101.67.42|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13661300 (13M) [application/pdf]
Saving to: ‘/Users/jatinder.singh/drive/git/Agentic_AI/agentic_ai/Multimodel_RAG_report_analysis/llama2.pdf’

/Users/jatinder.sin 100%[===================>]  13.03M  6.00MB/s    in 2.2s    

2025-07-04 17:19:25 (6.00 MB/s) - ‘/Users/jatinder.singh/drive/git/Agentic_AI/agentic_ai/Multimodel_RAG_report_analysis/llama2.pdf’ saved [13661300/13661300]



In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [5]:
pdf_loadder = PyPDFLoader("/Users/jatinder.singh/drive/git/Agentic_AI/agentic_ai/Multimodel_RAG_report_analysis/llama2.pdf")
docs = pdf_loadder.load()

In [6]:
from langchain_text_splitters import CharacterTextSplitter
from unstructured.partition.pdf import partition_pdf

def extract_pdf_elements(path, frame):
  return partition_pdf(
      filename=path + frame,
      extract_images_in_pdf=False,
      infer_table_structure=True,
      chunking_strategy="by_title",
      max_characters=4000,
      new_after_n_char=3800,
      combine_text_under_n_chars=2000,
      image_output_dir_path=path)

def categorize_elements(raw_pdf_elements):
    """
    Categorize extracted elements from a PDF into tables and texts.
    raw_pdf_elements: List of unstructured.documents.elements
    """
    tables = []
    texts = []
    for element in raw_pdf_elements:
        if "unstructured.documents.elements.Table" in str(type(element)):
            tables.append(str(element))
        elif "unstructured.documents.elements.CompositeElement" in str(type(element)):
            texts.append(str(element))
    return texts, tables
# File path
fpath = "/Users/jatinder.singh/drive/git/Agentic_AI/agentic_ai/Multimodel_RAG_report_analysis/"
fname = "llama2.pdf"

# Get elements
raw_pdf_elements = extract_pdf_elements(fpath, fname)

# Get text, tables
texts, tables = categorize_elements(raw_pdf_elements)

# Optional: Enforce a specific token size for texts
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=4000, chunk_overlap=0
)
joined_texts = " ".join(texts)
texts_4k_token = text_splitter.split_text(joined_texts)


/opt/homebrew/Caskroom/miniconda/base/envs/agentic/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


# Generate summaries of text elements
# def generate_text_summaries_open_ai(texts, tables, summarize_texts=False):
#     """
#     Summarize text elements
#     texts: List of str
#     tables: List of str
#     summarize_texts: Bool to summarize texts
#     """

#     # Prompt
#     prompt_text = """You are an assistant tasked with summarizing tables and text for retrieval. \
#     These summaries will be embedded and used to retrieve the raw text or table elements. \
#     Give a concise summary of the table or text that is well optimized for retrieval. Table or text: {element} """
#     prompt = ChatPromptTemplate.from_template(prompt_text)

#     # Text summary chain
#     model = ChatOpenAI(temperature=0, model="gpt-4")
#     summarize_chain = {"element": lambda x: x} | prompt | model | StrOutputParser()

#     # Initialize empty summaries
#     text_summaries = []
#     table_summaries = []

#     # Apply to text if texts are provided and summarization is requested
#     if texts and summarize_texts:
#         text_summaries = summarize_chain.batch(texts, {"max_concurrency": 5})
#     elif texts:
#         text_summaries = texts

#     # Apply to tables if tables are provided
#     if tables:
#         table_summaries = summarize_chain.batch(tables, {"max_concurrency": 5})

#     return text_summaries, table_summaries

from transformers import pipeline

# Use summarization pipeline
pipe = pipeline("summarization", model="adept/fuyu-8b")

def generate_text_summaries(texts, tables, summarize_texts=False):
    text_summaries = []
    table_summaries = []

    # Summarize texts
    if texts and summarize_texts:
        for text in texts:
            summary = pipe(text, max_length=128, min_length=30, do_sample=False)
            text_summaries.append(summary[0]['summary_text'])
    elif texts:
        text_summaries = texts

    # Summarize tables (if you want to treat tables as text)
    if tables:
        for table in tables:
            summary = pipe(table, max_length=128, min_length=30, do_sample=False)
            table_summaries.append(summary[0]['summary_text'])

    return text_summaries, table_summaries
# Get text, table summaries
text_summaries, table_summaries = generate_text_summaries(
    texts_4k_token, tables, summarize_texts=True
)

Fetching 2 files:   0%|          | 0/2 [16:30<?, ?it/s]


In [ ]:
from transformers import pipeline

# Use summarization pipeline
pipe = pipeline("summarization", model="adept/fuyu-8b")

/opt/homebrew/Caskroom/miniconda/base/envs/agentic/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 2 files:   0%|          | 0/2 [00:45<?, ?it/s]
